# 00 · Datos de origen: DENUE 2020 y 2025, Censo 2020

**Proyecto reducido.** Solo se usan las dos ediciones del DENUE que incorporan un Censo Económico, porque son las únicas en las que el directorio registra el cambio real de los negocios:

| Edición | Censo Económico que incorpora |
|---|---|
| **04/2020** | 2019 |
| **05/2025** | 2024 |

Entre censos el INEGI solo actualiza parcialmente a los negocios pequeños, así que cualquier otra edición mediría el calendario de actualización, no la calle.

También se usa el **Censo de Población y Vivienda 2020** por manzana y el **Marco Geoestadístico 2020** (centroides de manzana).

**Para no descargar dos veces:** si el proyecto anterior (`tesis_queretaro/data`) ya tiene estos archivos, se copian. Si no, se descargan del INEGI.

## 0. Librerías y rutas

In [ ]:
from pathlib import Path
import io, shutil, tempfile, time, zipfile, urllib.request
import pandas as pd

ROOT = Path.cwd().parent                  # proyecto_reducido/
PADRE = ROOT.parent                       # tesis_queretaro/ (proyecto anterior)
RAW, PROC = ROOT / "data/raw", ROOT / "data/processed"
for p in (RAW, PROC, ROOT / "data/validacion", ROOT / "figures"):
    p.mkdir(parents=True, exist_ok=True)
TMP = Path(tempfile.gettempdir()) / "denue_tmp"     # fuera de Dropbox: no sincroniza los zips
TMP.mkdir(exist_ok=True)
BASE_URL = "https://www.inegi.org.mx/contenidos/masiva/denue"

## 1. Configuración

`EDICIONES`: etiqueta `AAAA-MM` → carpeta del INEGI. Solo las dos ediciones con Censo Económico.

In [ ]:
EDICIONES = {
    "2020-04": "2020_04",   # incorpora el Censo Económico 2019
    "2025-05": "2025_05",   # incorpora el Censo Económico 2024
}

# Los 25 archivos por sector que publica el INEGI
SECTORES = ["11", "21", "22", "23", "31-33", "43", "46111", "46112-46311", "46321-46531",
            "46591-46911", "48-49", "51", "52", "53", "54", "55", "56", "61", "62",
            "71", "72_1", "72_2", "81_1", "81_2", "93"]

ARCHIVO_CENSO = "ageb_mza_urbana_22_cpv2020_csv.zip"
URL_CENSO = f"https://www.inegi.org.mx/contenidos/programas/ccpv/2020/datosabiertos/ageb_manzana/{ARCHIVO_CENSO}"

## 2. Funciones

- `copiar_del_padre`: si el archivo ya existe en el proyecto anterior, lo copia y evita la descarga.
- `url_archivo`: arma la liga del INEGI, por ejemplo `2025_05/denue_00_62_0525_csv.zip`.
- `filtrar_queretaro`: lee cada CSV por bloques y se queda solo con Querétaro (`cve_ent = 22`).

In [ ]:
def copiar_del_padre(relativo, destino):
    origen = PADRE / relativo
    if destino.exists():
        print(f"Ya existe: {destino.name}")
        return True
    if origen.exists():
        shutil.copy2(origen, destino)
        print(f"Copiado del proyecto anterior: {destino.name}")
        return True
    return False

def url_archivo(carpeta, sector):
    aaaa, mm = carpeta.split("_")
    return f"{BASE_URL}/{carpeta}/denue_00_{sector}_{mm}{aaaa[2:]}_csv.zip"

def descargar(url, destino, intentos=3):
    for i in range(intentos):
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
            with urllib.request.urlopen(req, timeout=300) as r, open(destino, "wb") as f:
                while bloque := r.read(1 << 20):
                    f.write(bloque)
            return destino
        except Exception as e:
            print(f"    intento {i + 1} falló: {e}")
            time.sleep(5)
    raise RuntimeError(f"No se pudo descargar {url}")

def detectar_encoding(z, info):
    with z.open(info) as f:
        muestra = f.read(2_000_000)
    muestra = muestra[: muestra.rfind(b"\n")]
    try:
        muestra.decode("utf-8")
        return "utf-8"
    except UnicodeDecodeError:
        return "latin-1"

def filtrar_queretaro(ruta_zip):
    with zipfile.ZipFile(ruta_zip) as z:
        info = max((i for i in z.infolist() if i.filename.lower().endswith(".csv")), key=lambda i: i.file_size)
        enc = detectar_encoding(z, info)
        partes = []
        with z.open(info) as f:
            for bloque in pd.read_csv(io.TextIOWrapper(f, encoding=enc), dtype=str, chunksize=200_000):
                bloque.columns = bloque.columns.str.strip().str.lower()
                partes.append(bloque[pd.to_numeric(bloque["cve_ent"], errors="coerce") == 22])
    return pd.concat(partes, ignore_index=True)

## 3. DENUE: copiar o descargar las dos ediciones

In [ ]:
for etiqueta, carpeta in EDICIONES.items():
    salida = RAW / f"denue_qro_{etiqueta}.parquet"
    if copiar_del_padre(f"data/raw/denue_qro_{etiqueta}.parquet", salida):
        continue
    print(f"=== Descargando {etiqueta} (unos minutos) ===")
    partes = []
    for s in SECTORES:
        url = url_archivo(carpeta, s)
        tmp = TMP / url.split("/")[-1]
        descargar(url, tmp)
        partes.append(filtrar_queretaro(tmp))
        tmp.unlink()
        print(f"  {s:>12}: {len(partes[-1]):>7,} registros")
    pd.concat(partes, ignore_index=True).to_parquet(salida, index=False)

## 4. Censo 2020 y centroides de manzana

- El CSV del Censo por manzana se copia o se descarga a `data/raw/`.
- Los centroides de manzana (`manzanas_zmq_2020.parquet`) se copian del proyecto anterior si existen; si no, el notebook 02 los construye desde el Marco Geoestadístico.

In [ ]:
if not copiar_del_padre(f"data/raw/{ARCHIVO_CENSO}", RAW / ARCHIVO_CENSO):
    print("Descargando el Censo 2020 por manzana…")
    descargar(URL_CENSO, RAW / ARCHIVO_CENSO)

if not copiar_del_padre("data/processed/manzanas_zmq_2020.parquet", PROC / "manzanas_zmq_2020.parquet"):
    print("No hay centroides de manzana en el proyecto anterior: el notebook 02 los construye.")

## 5. Verificación

In [ ]:
for p in sorted(RAW.glob("denue_qro_*.parquet")):
    df = pd.read_parquet(p, columns=["cve_mun"])
    print(f"{p.name}: {len(df):,} registros de Querétaro · {p.stat().st_size / 1e6:.1f} MB")
print("Censo:", (RAW / ARCHIVO_CENSO).exists(), "· Centroides:", (PROC / "manzanas_zmq_2020.parquet").exists())